In [1]:
import sys
from pathlib import Path
import importlib

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data import load_dataset, split_dataset
from src.features import extract_frame_features, extract_feature_vector, build_feature_matrix
from src.models import fit_scaler, train_logistic_regression, train_svm, train_mlp, predict_mlp
# import src.evaluation
# importlib.reload(src.evaluation)
from src.evaluation import evaluate_model


# Loading the dataset
df = load_dataset(project_root /"data" / "IRMAS-TrainingData" / "IRMAS-TrainingData")

print(df.head())
print(df["label"].value_counts())
print("Total: ", len(df))

# Creating split
train_df, val_df, test_df = split_dataset(df)

print(len(train_df), len(val_df), len(test_df))

print(train_df["label"].value_counts(normalize=True))
print(val_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

print(len(train_df)+len(val_df)+len(test_df))
print(train_df["label"].nunique())
print(val_df["label"].nunique())
print(test_df["label"].nunique())

                                            filepath  label label_short
0  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
1  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
2  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
3  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
4  c:\Users\nimac\KTH\DT2470 Music Informatics\mu...  cello         cel
label
voice              778
electric_guitar    760
piano              721
organ              682
acoustic_guitar    637
saxophone          626
violin             580
trumpet            577
clarinet           505
flute              451
cello              388
Name: count, dtype: int64
Total:  6705
4693 1006 1006
label
voice              0.115917
electric_guitar    0.113360
piano              0.107607
organ              0.101641
acoustic_guitar    0.095035
saxophone          0.093330
violin             0.086512
trumpet            0.086086
clarinet           0.0752

In [2]:
# Feature selection: Mean pooled
# MFCC
x_train_mfcc_mean, y_train = build_feature_matrix(train_df, selected_feature="mfcc", pooling="mean")
x_val_mfcc_mean, y_val = build_feature_matrix(val_df, selected_feature="mfcc", pooling="mean")
x_test_mfcc_mean, y_test = build_feature_matrix(test_df, selected_feature="mfcc", pooling="mean")

# print(x_train.shape, y_train.shape)
# print(x_val.shape, y_val.shape)
# print(x_test.shape, y_test.shape)

# Additional features: spectral centroid + ZCR
x_train_low_level_mean, _ = build_feature_matrix(train_df, selected_feature="low_level", pooling="mean")
x_val_low_level_mean, _ = build_feature_matrix(val_df, selected_feature="low_level", pooling="mean")
x_test_low_level_mean, _ = build_feature_matrix(test_df, selected_feature="low_level", pooling="mean")

# Combined: MFCC + spectral centroid + ZCR
x_train_combined_mean, _ = build_feature_matrix(train_df, selected_feature="combined", pooling="mean")
x_val_combined_mean, _ = build_feature_matrix(val_df, selected_feature="combined", pooling="mean")
x_test_combined_mean, _ = build_feature_matrix(test_df, selected_feature="combined", pooling="mean")

# Labels for MLP (integer labels)
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

In [3]:
# Feature and model comparison experiment: Training models on mean pooled features
# Normalize features
x_train_mfcc_mean_scaled, scaler_mfcc_mean = fit_scaler(x_train_mfcc_mean)
x_val_mfcc_mean_scaled = scaler_mfcc_mean.transform(x_val_mfcc_mean)

x_train_low_level_mean_scaled, scaler_low_level_mean = fit_scaler(x_train_low_level_mean)
x_val_low_level_mean_scaled = scaler_low_level_mean.transform(x_val_low_level_mean)

x_train_combined_mean_scaled, scaler_combined_mean = fit_scaler(x_train_combined_mean)
x_val_combined_mean_scaled = scaler_combined_mean.transform(x_val_combined_mean)

# Model training
# MFCC
lr_mfcc_mean = train_logistic_regression(x_train_mfcc_mean_scaled, y_train) # This is our baseline
svm_mfcc_mean = train_svm(x_train_mfcc_mean_scaled, y_train)
mlp_mfcc_mean, mlp_mfcc_mean_train_losses, mlp_mfcc_mean_val_losses = train_mlp(x_train_mfcc_mean_scaled, 
                                y_train_encoded, 
                                x_val_mfcc_mean_scaled, 
                                y_val_encoded, 
                                input_dim=x_train_mfcc_mean_scaled.shape[1])

# Spectral centroids + ZCR
lr_low_level_mean = train_logistic_regression(x_train_low_level_mean_scaled, y_train)
svm_low_level_mean = train_svm(x_train_low_level_mean_scaled, y_train)
mlp_low_level_mean, mlp_low_level_mean_train_losses, mlp_low_level_mean_val_losses = train_mlp(x_train_low_level_mean_scaled, 
                                y_train_encoded, 
                                x_val_low_level_mean_scaled, 
                                y_val_encoded, 
                                input_dim=x_train_low_level_mean_scaled.shape[1])

# Combined
lr_combined_mean = train_logistic_regression(x_train_combined_mean_scaled, y_train)
svm_combined_mean = train_svm(x_train_combined_mean_scaled, y_train)
mlp_combined_mean, mlp_combined_mean_train_losses, mlp_combined_mean_val_losses = train_mlp(x_train_combined_mean_scaled, 
                                y_train_encoded, 
                                x_val_combined_mean_scaled, 
                                y_val_encoded, 
                                input_dim=x_train_combined_mean_scaled.shape[1])


# Predictions
# MFCC
lr_mfcc_mean_pred = lr_mfcc_mean.predict(x_val_mfcc_mean_scaled)
svm_mfcc_mean_pred = svm_mfcc_mean.predict(x_val_mfcc_mean_scaled)
mlp_mfcc_mean_pred_encoded = predict_mlp(mlp_mfcc_mean, x_val_mfcc_mean_scaled)
mlp_mfcc_mean_pred = label_encoder.inverse_transform(mlp_mfcc_mean_pred_encoded)

# Spectral centroids + ZCR
lr_low_level_mean_pred = lr_low_level_mean.predict(x_val_low_level_mean_scaled)
svm_low_level_mean_pred = svm_low_level_mean.predict(x_val_low_level_mean_scaled)
mlp_low_level_mean_pred_encoded = predict_mlp(mlp_low_level_mean, x_val_low_level_mean_scaled)
mlp_low_level_mean_pred = label_encoder.inverse_transform(mlp_low_level_mean_pred_encoded)

# Combined
lr_combined_mean_pred = lr_combined_mean.predict(x_val_combined_mean_scaled)
svm_combined_mean_pred = svm_combined_mean.predict(x_val_combined_mean_scaled)
mlp_combined_mean_pred_encoded = predict_mlp(mlp_combined_mean, x_val_combined_mean_scaled)
mlp_combined_mean_pred = label_encoder.inverse_transform(mlp_combined_mean_pred_encoded)

In [4]:
# Evaluation: Mean pooled features
print("MFCC - Logistic Regression")
lr_mfcc_mean_results = evaluate_model(y_val, lr_mfcc_mean_pred)

print("\nMFCC - SVM")
svm_mfcc_mean_results = evaluate_model(y_val, svm_mfcc_mean_pred)

print("\nMFCC - MLP")
mlp_mfcc_mean_results = evaluate_model(y_val, mlp_mfcc_mean_pred)

print("\nLow level - Logistic regression")
lr_low_level_mean_results = evaluate_model(y_val, lr_low_level_mean_pred)

print("\nLow level - SVM")
svm_low_level_mean_results = evaluate_model(y_val, svm_low_level_mean_pred)

print("\nLow level - MLP")
mlp_low_level_mean_results = evaluate_model(y_val, mlp_low_level_mean_pred)

print("\nCombined - Logistic Regression")
lr_combined_mean_results = evaluate_model(y_val, lr_combined_mean_pred)

print("\nCombined - SVM")
svm_combined_mean_results = evaluate_model(y_val, svm_combined_mean_pred) 

print("\nCombined - MLP")
mlp_combined_mean_results = evaluate_model(y_val, mlp_combined_mean_pred)

# Save it as a pandas dataframe
import pandas as pd

results_table = pd.DataFrame([
    {
        "Feature set": "MFCC",
        "Pooling": "Mean",
        "Model": "Logistic Regression",
        "Accuracy": lr_mfcc_mean_results["accuracy"],
        "Macro F1": lr_mfcc_mean_results["macro_f1"]
    },
    {
        "Feature set": "MFCC",
        "Pooling": "Mean",
        "Model": "SVM",
        "Accuracy": svm_mfcc_mean_results["accuracy"],
        "Macro F1": svm_mfcc_mean_results["macro_f1"]
    },
    {
        "Feature set": "MFCC",
        "Pooling": "Mean",
        "Model": "MLP",
        "Accuracy": mlp_mfcc_mean_results["accuracy"],
        "Macro F1": mlp_mfcc_mean_results["macro_f1"]
    },
    {
        "Feature set": "Low-level",
        "Pooling": "Mean",
        "Model": "Logistic Regression",
        "Accuracy": lr_low_level_mean_results["accuracy"],
        "Macro F1": lr_low_level_mean_results["macro_f1"]
    },
    {
        "Feature set": "Low-level",
        "Pooling": "Mean",
        "Model": "SVM",
        "Accuracy": svm_low_level_mean_results["accuracy"],
        "Macro F1": svm_low_level_mean_results["macro_f1"]
    },
    {
        "Feature set": "Low-level",
        "Pooling": "Mean",
        "Model": "MLP",
        "Accuracy": mlp_low_level_mean_results["accuracy"],
        "Macro F1": mlp_low_level_mean_results["macro_f1"]
    },
    {
        "Feature set": "Combined",
        "Pooling": "Mean",
        "Model": "Logistic Regression",
        "Accuracy": lr_combined_mean_results["accuracy"],
        "Macro F1": lr_combined_mean_results["macro_f1"]
    },
    {
        "Feature set": "Combined",
        "Pooling": "Mean",
        "Model": "SVM",
        "Accuracy": svm_combined_mean_results["accuracy"],
        "Macro F1": svm_combined_mean_results["macro_f1"]
    },
    {
        "Feature set": "Combined",
        "Pooling": "Mean",
        "Model": "MLP",
        "Accuracy": mlp_combined_mean_results["accuracy"],
        "Macro F1": mlp_combined_mean_results["macro_f1"]
    }
])

MFCC - Logistic Regression
Accuracy: 0.3907
Macro F1: 0.3860

                 precision    recall  f1-score   support

acoustic_guitar       0.37      0.43      0.40        95
          cello       0.52      0.43      0.47        58
       clarinet       0.51      0.39      0.44        76
electric_guitar       0.36      0.43      0.39       114
          flute       0.53      0.25      0.34        68
          organ       0.36      0.47      0.41       102
          piano       0.37      0.44      0.40       108
      saxophone       0.33      0.18      0.23        94
        trumpet       0.42      0.38      0.40        87
         violin       0.34      0.29      0.31        87
          voice       0.39      0.51      0.44       117

       accuracy                           0.39      1006
      macro avg       0.41      0.38      0.39      1006
   weighted avg       0.40      0.39      0.39      1006


MFCC - SVM
Accuracy: 0.5010
Macro F1: 0.4921

                 precision    rec

c:\Users\nimac\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nimac\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nimac\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [5]:
# Display results table from baseline experiment
results_table

,Feature set,Pooling,Model,Accuracy,Macro F1
0,MFCC,Mean,Logistic Regression,0.390656,0.385961
1,MFCC,Mean,SVM,0.500994,0.492052
2,MFCC,Mean,MLP,0.499006,0.492478
3,Low-level,Mean,Logistic Regression,0.221670,0.151762
4,Low-level,Mean,SVM,0.262425,0.193689
5,Low-level,Mean,MLP,0.258449,0.214085
6,Combined,Mean,Logistic Regression,0.402584,0.396317
7,Combined,Mean,SVM,0.513917,0.505135
8,Combined,Mean,MLP,0.496024,0.486369
